In [1]:
import json
import os
from pathlib import Path
from pprint import pprint
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import gaussian_filter, uniform_filter
import xarray as xr

dirpath_repo_root = Path().resolve().parents[1]
dirpath_self = Path().resolve().parent
sys.path.append(str(dirpath_repo_root))
sys.path.append(str(dirpath_self))

from analysis.ou_tuning import batch_utils, json_utils

### Common parameters

In [2]:
exp_name = 'batch_rxbkg_unconn_state1_mech1/pops_inpsur_newsec_var_drxe_rxi'

dirpath_cfg = dirpath_repo_root / 'exp_configs' / exp_name
dirpath_res = dirpath_repo_root / 'exp_results' / exp_name

fname_data_templ = 'result_{job:05d}_*.json'

vars = {
    'rate': 'rates', 'cv': 'cvs',
    'vmin': 'v_med_min', 'vmax': 'v_med_max',
    'vavg': 'v_thresh_avg', 'vstd': 'v_thresh_std'
}

### Helper functions

In [3]:
def _get_fpath_by_templ(dirpath: Path, fname_templ: str) -> str:
    files = list(dirpath.glob(fname_templ))
    if len(files) != 1:
        print(f'Path for search: {str(dirpath)}')
        print(f'Name template: {fname_templ}')
        raise RuntimeError('Should be exactly one filename match')
    return files[0]

In [4]:
def load_job_idx(exp_name_sub):
    """Extract (param_grid -> job_id) xarray from cfg folder. """
    job_idx_xr = batch_utils.extract_batch_params_to_xr(
        dirpath_res / exp_name_sub / 'cfg',
        cfg_param_fields={'drxe_pos': 'drxe_num', 'rxi_pos': 'rxi_num'},
        fname_cfg_templ='cfg_*.json',
        job_pos_in_fname=1
    )
    if job_idx_xr.size == 0:
        raise RuntimeError('No cfg files found')
    return job_idx_xr


def load_regions(exp_label):
    """Load (drxe, rxi) regions. """
    fpath_rgn = dirpath_cfg / 'regions' / f'regions_{exp_label}.json'
    with open(fpath_rgn, 'r') as fid:
        regions = json.load(fid)['regions']
    return regions


def load_target_rates():
    fpath_rates = dirpath_cfg / 'target_state_1.csv'
    df = pd.read_csv(fpath_rates).set_index('pop_name')
    return df['target_rate'].to_dict()


def load_wx():
    fname_wx = 'wx_target_psp_0.5_soma_mech1_vrest_pyr_-70.json'
    with open(dirpath_cfg  /fname_wx, 'r') as fid:
        wx = json.load(fid)['wx_target']
    return wx


def alloc_datasets(pops_used, vars, job_idx_xr, regions):
    """Allocate xarray Dataset for each population. """
    n_drxe = job_idx_xr.sizes['drxe_pos']
    n_rxi = job_idx_xr.sizes['rxi_pos']

    X = {}
    for pop in pops_used:
        # Create a Dataset
        X_ = {}
        for v in vars:
            X_[v] = xr.full_like(job_idx_xr, np.nan, dtype=np.float64)
        X_ = xr.Dataset(X_)

        # Region
        rgn = regions[pop]
        rxe0, rxi0 = rgn['rxe0'], rgn['rxi0']
        drxe, drxi = rgn['drxe'], rgn['drxi']
        drxe_vals = np.linspace(-drxe, 0, n_drxe)
        rxi_vals = np.linspace(rxi0, rxi0 + drxi, n_rxi)

        # Add drxe and rxi coords
        X_ = X_.assign_coords(
            drxe=('drxe_pos', drxe_vals),
            rxi=('rxi_pos', rxi_vals),
        )        
        X[pop] = X_

    return X


In [5]:
def load_data(X, job_idx_xr, exp_name_sub, vars, pops_used):
    """Load batch results into pre-allocated X (in-place). """
    
    # Iterate over all cells in job_idx_xr and fill X
    for idx in np.ndindex(job_idx_xr.shape):
        job_id = job_idx_xr.values[idx]
        if np.isnan(job_id):
            continue
        job_id = int(job_id)
        sel = {dim: idx[i] for i, dim in enumerate(job_idx_xr.dims)}
        
        # Generate file path for the job data
        fname_data = fname_data_templ.format(job=job_id)
        dirpath_data = dirpath_res / exp_name_sub / 'results'
        fpath_data = _get_fpath_by_templ(dirpath_data, fname_data)
        #print(job_id, fpath_data.name)

        # Load data and assign it to the corresponding slices in X
        with open(fpath_data, 'r') as fid:
            res = json.load(fid)
        for v, v_res in vars.items():
            for pop in pops_used:
                if pop in res[v_res]:
                    X[pop][v][sel] = res[v_res][pop]

    for pop in pops_used:
        X[pop] = X[pop].swap_dims({
            'drxe_pos': 'drxe',
            'rxi_pos': 'rxi',
        })

In [6]:
def upsample_and_smooth(da, coords_new, win):
    """Upsample and smooth while preserving the original NaN mask footprint. """

    da_up = da.interp(**coords_new)
    v = np.asarray(da_up.values, dtype=float)

    mask0 = da.notnull().astype(float)
    #mask_up = mask0.interp(**coords_new, method='nearest').astype(bool)
    mask_up = mask0.interp(**coords_new, method='nearest').values.astype(bool)

    #num = da_up.fillna(0).rolling(**win, center=True, min_periods=1).sum()
    #den = da_up.notnull().astype(float).rolling(**win, center=True, min_periods=1).sum()

    valid = np.isfinite(v).astype(float)
    v0 = np.nan_to_num(v, nan=0.0)

    wy = win[da_up.dims[0]]
    wx = win[da_up.dims[1]]

    num = uniform_filter(v0, size=(wy, wx), mode='constant', cval=0.0)
    den = uniform_filter(valid, size=(wy, wx), mode='constant', cval=0.0)

    #out = num / den
    #out = out.where(mask_up)
    out = np.divide(num, den, where=(den != 0))
    out[den == 0] = np.nan
    out[~mask_up] = np.nan

    #return out
    return xr.DataArray(out, coords=da_up.coords, dims=da_up.dims)


def upsample_and_smooth_gauss(da, coords_new, sigma):
    """Upsample and Gaussian-smooth while preserving the original NaN mask footprint."""

    da_up = da.interp(**coords_new)
    v = np.asarray(da_up.values, dtype=float)

    mask0 = da.notnull().astype(float)
    mask_up = mask0.interp(**coords_new, method='nearest').values.astype(bool)

    if sigma == 0:
        v[~mask_up] = np.nan
        return xr.DataArray(v, coords=da_up.coords, dims=da_up.dims)    

    valid = np.isfinite(v).astype(float)
    v0 = np.nan_to_num(v, nan=0.0)
    
    sx, sy = sigma, sigma

    num = gaussian_filter(v0, sigma=(sy, sx), mode='constant', cval=0.0)
    den = gaussian_filter(valid, sigma=(sy, sx), mode='constant', cval=0.0)

    out = np.empty_like(num)
    out.fill(np.nan)
    np.divide(num, den, out=out, where=(den != 0))
    out[~mask_up] = np.nan

    return xr.DataArray(out, coords=da_up.coords, dims=da_up.dims)


def find_contour(da, level, xname, yname):
    """Find and concatenate all contour segments. """
    x = da.coords[xname].values
    y = da.coords[yname].values
    Z = np.ma.masked_invalid(da.transpose(yname, xname).values)

    Xg, Yg = np.meshgrid(x, y)

    fig, ax = plt.subplots()
    cs = ax.contour(Xg, Yg, Z, levels=[level])
    plt.close(fig)

    segs = [seg for seg in cs.allsegs[0] if len(seg) > 0]
    pts = np.vstack(segs)
    seg_id = np.concatenate([np.full(len(seg), i) for i, seg in enumerate(segs)])

    return {
        'xy': pts,
        xname: pts[:, 0],
        yname: pts[:, 1],
        'seg_id': seg_id,
    }


def interp_on_points(da, pts, xname, yname):
    """Interpolate a variable on arbitrary points. """
    x = da.coords[xname].values
    y = da.coords[yname].values
    Z = np.asarray(da.transpose(yname, xname).values, dtype=float)

    interp = RegularGridInterpolator(
        (y, x), Z,
        method='linear',
        bounds_error=False,
        fill_value=np.nan
    )
    return interp(np.c_[pts[:, 1], pts[:, 0]])


def split_by_seg(arr, seg_id):
    """Split a contour-aligned array into contour segments. """
    return [arr[seg_id == i] for i in np.unique(seg_id)]


def get_coord_bounds(X, xname, yname, k):
    x0 = float(X[xname].min()) * k
    x1 = float(X[xname].max()) * k
    y0 = float(X[yname].min()) * k
    y1 = float(X[yname].max()) * k
    return x0, x1, y0, y1


In [7]:
def prepare_maps(X, pop, r0, vars, n_grid=200, sm_win=71,
                 xname='drxe', yname='rxi', k=1):

    vars_used = list(vars.keys())
    X_pop = X[pop]

    # Compute coordinate bounds
    x0, x1, y0, y1 = get_coord_bounds(X_pop, xname, yname, k)

    # Build upsampled grid
    coords_new = {
        xname: np.linspace(x0, x1, n_grid),
        yname: np.linspace(y0, y1, n_grid),
    }

    # Define smoothing window
    win = {xname: sm_win, yname: sm_win}   # for box filter
    sigma = sm_win / 3.5   # for gaussian window

    # Upsample and smooth all variables on the same grid
    X_up = {
        #v: upsample_and_smooth(X_pop[v], coords_new, win)
        v: upsample_and_smooth_gauss(X_pop[v], coords_new, sigma)
        for v in vars_used
    }

    # Find the r=r0 contour on the upsampled rate map
    contour = find_contour(X_up['rate'], r0, xname, yname)

    # Sample all variables on the contour points
    for v in vars_used:
        contour[v] = interp_on_points(X_up[v], contour['xy'], xname, yname)

    """ # Find the contour point with maximal CV
    cv_ = contour['cv']
    if np.all(np.isnan(cv_)):
        id_sel = int(len(cv_) / 2)
    else:
        id_sel = np.nanargmax(cv_)
    x_sel = contour[xname][id_sel]
    y_sel = contour[yname][id_sel]
    cv_sel = cv_[id_sel] """

    # Find contour midpoint
    id_sel = int(len(contour['rate']) / 2)
    x_sel = contour[xname][id_sel]
    y_sel = contour[yname][id_sel]
    cv_sel = contour['cv'][id_sel]

    # Collect the results
    res = {
        'X_up': X_up, 'contour': contour,
        'x_sel': x_sel, 'y_sel': y_sel,
        'id_sel': id_sel, 'cv_sel': cv_sel,
        'par': {
            'pop': pop, 'r0': r0, 'xname': xname, 'yname': yname,
            'k': k, 'n_grid': n_grid, 'sm_win': sm_win
        }
    }
    return res

In [8]:
def calc_rxe(rgn, rxi, drxe):
    rxe0, rxi0 = rgn['rxe0'], rgn['rxi0']
    k = rgn['rxi_rxe_slope']
    rxe = rxe0 + (rxi - rxi0) / k + drxe
    return np.maximum(rxe, 0)

In [9]:
def plot_2d_map(
        fig, ax,
        maps, var,
        xlabel=True, ylabel=True,
        title=True, legend=True,
        transpose=True,
        show_map=True,
        contour_style='k-'
        ):

    # Map, contour, chosen point
    X = maps['X_up'][var]
    contour = maps['contour']
    x_sel, y_sel = maps['x_sel'], maps['y_sel']

    par = maps['par']
    pop, r0 = par['pop'], par['r0']
    xname, yname, k = par['xname'], par['yname'], par['k']

    if transpose:
        xname, yname = yname, xname
        x_sel, y_sel = y_sel, x_sel

    # Compute coordinate bounds
    x0, x1, y0, y1 = get_coord_bounds(X, xname, yname, k)

    # Plot 2D map
    if show_map:
        im = ax.imshow(
            X.transpose(xname, yname).values.T,
            extent=(x0, x1, y0, y1),
            aspect='auto',
            origin='lower',
            interpolation='bicubic'
        )

    # Plot contour: r=r0
    for xs, ys in zip(
            split_by_seg(contour[xname], contour['seg_id']),
            split_by_seg(contour[yname], contour['seg_id'])):
        ax.plot(xs, ys, contour_style, lw=2)

    # Plot chosen point
    ax.plot(x_sel, y_sel, 'ro')

    if title:
        ax.set_title(f'{pop}, {var}')
    if xlabel:
        ax.set_xlabel(xname)
    if ylabel:
        ax.set_ylabel(yname)
    if legend:
        ax.legend([f'r={r0:.01f} Hz'])
    
    ax.ticklabel_format(style='sci', axis='both', scilimits=(0,0))
    
    if show_map:
        fig.colorbar(im, ax=ax, shrink=0.8)

In [10]:
def plot_1d_map(
        fig, ax,
        maps, var,
        xlabel=True, ylabel=True,
        title=True, legend=True
        ):

    # Map, contour, chosen point
    X = maps['X_up'][var]
    contour = maps['contour']
    id_sel = maps['id_sel']

    par = maps['par']
    pop, r0 = par['pop'], par['r0']
    yname = par['yname']

    xc = contour[yname]
    xc_max = xc[id_sel]

    # Plot variable along the contour
    for xs, ys in zip(
            split_by_seg(xc, contour['seg_id']),
            split_by_seg(contour[var], contour['seg_id'])):
        ax.plot(xs, ys, 'k.')

    # Plot chosen point
    ax.plot(xc_max, contour[var][id_sel], 'ro')

    if title:
        ax.set_title(f'{pop}, {var}')
    if xlabel:
        ax.set_xlabel(yname)
    if ylabel:
        ax.set_ylabel(var)
    if legend:
        #ax.legend([f'{var}, r={r0:.01f} Hz'])
        ax.legend([var])
    
    if var == 'rate':
        ax.set_ylim(r0 - 1, r0 + 1)

In [11]:
def plot_maps(
        fig, axs, maps,
        transpose=True,
        show_2d_maps=True,
        contour_style='k-',
        vars_vis=None    
        ):

    vars_vis = vars_vis or ['rate', 'cv', 'vavg']
    nvars = len(vars_vis)

    for n, var in enumerate(vars_vis):
        # 2D map
        #plt.subplot(2, nvars, n + 1)
        plot_2d_map(
            fig, axs[0, n], maps, var,
            xlabel=True, ylabel=(n == 0),
            title=True, legend=(n == 0),
            transpose=transpose,
            show_map=show_2d_maps,
            contour_style=contour_style
        )
        # 1D plot
        #plt.subplot(2, nvars, nvars + n + 1)
        plot_1d_map(
            fig, axs[1, n], maps, var,
            xlabel=True, ylabel=False,
            title=False, legend=True
        )

### Main

In [12]:
proc_label = 'ctx'

pop_groups = {
    'it2': ['IT2'],
    'it3': ['IT3'],
    'it4': ['ITP4', 'ITS4'],
    'it5': ['IT5A', 'IT5B'],
    'it6': ['IT6'],
    'ct': ['CT5A', 'CT5B', 'CT6'],
    'pt5b': ['PT5B'],
    'pv': ['PV2', 'PV3', 'PV4', 'PV5A', 'PV5B', 'PV6'],
    'som': ['SOM2', 'SOM3', 'SOM4', 'SOM5A', 'SOM5B', 'SOM6'],
    'vip': ['VIP2', 'VIP3', 'VIP4', 'VIP5A', 'VIP5B', 'VIP6'],
    'ngf': ['NGF1', 'NGF2', 'NGF3', 'NGF4', 'NGF5A', 'NGF5B', 'NGF6']
}

exp_name_sub_group = '2d_20x20_psp_0.5_xsec_soma_soma'
exp_name_sub_templ = (
    'exp_%_sz_20_20_t_7.0_12.0_wx_psp_0.5_wmult_0.25_ee_0.5_lsec_0_xsec_soma_soma'
)

xe_sec, xi_sec = 'soma', 'soma'

sm_win = 21

dirpath_out = dirpath_res / exp_name_sub_group / f'rxsel_{proc_label}_sm_{sm_win}'
dirpath_figs = dirpath_out / f'map_figs'
os.makedirs(dirpath_figs, exist_ok=True)

def get_exp_name_sub(exp_label):
    exp_name_sub = exp_name_sub_templ.replace('%', exp_label)
    exp_name_sub = f'{exp_name_sub_group}/{exp_name_sub}'
    return exp_name_sub

target_rates = load_target_rates()
wx_info = load_wx()


In [13]:
def plot_slice(ax, R, maps, pop, r0):
    rxi_sel = maps['rxi_sel']
    X_up = maps['X_up']['rate']
    rr = R.sel(rxi=rxi_sel, drop=True, method='nearest')
    rr_sm = X_up.sel(rxi=rxi_sel, drop=True, method='nearest')

    ax.plot(rr.drxe, rr, '.-')
    ax.plot(rr_sm.drxe, rr_sm, '-')
    ax.axhline(r0, color='k', alpha=0.5)
    ax.set_xlabel('drxe')
    ax.set_ylabel('Rate')
    #ax.ylim(0, 10)
    ax.set_title(f'{pop}, rxi={rxi_sel:.01f}')

In [14]:
res = {}
rx_sel = []
X_all = {}

pop_groups_vis = pop_groups
#pop_groups_vis = {'it2': ['IT2']}

for exp_label, pops_used in pop_groups_vis.items():

    # Load batch data
    exp_name_sub = get_exp_name_sub(exp_label)
    job_idx_xr = load_job_idx(exp_name_sub)
    regions = load_regions(exp_label)
    X = alloc_datasets(pops_used, vars, job_idx_xr, regions)
    load_data(X, job_idx_xr, exp_name_sub, vars, pops_used)
    X_all[exp_label] = X

    for pop in pops_used:
        print(pop)

        # Target rate
        r0 = target_rates[pop]

        # Find r=r0 contour and select the point
        maps = prepare_maps(X, pop, r0, vars, sm_win=sm_win)
        drxe, rxi = maps['x_sel'], maps['y_sel']
        rxe = calc_rxe(regions[pop], rxi, drxe)
        maps['rxe_sel'], maps['rxi_sel'] = rxe, rxi
        cv = np.round(maps['cv_sel'], 1)

        # Find r=r0 contour (non-smoothed)
        maps0 = prepare_maps(X, pop, r0, vars, sm_win=0)

        # Plot maps
        fig, axs = plt.subplots(2, 3, figsize=(12, 8))
        plot_maps(fig, axs, maps)
        #plot_maps(fig, axs, maps)
        plot_maps(fig, axs, maps0, show_2d_maps=False, contour_style='k--')

        # Plot selected slice
        axs[1, 0].clear()
        plot_slice(axs[1, 0], X[pop]['rate'], maps, pop, r0)
        
        # Save the plots
        plt.savefig(dirpath_figs / f'{pop}.png', dpi=300)
        plt.close()

        # Append the results
        maps.pop('X_up'), maps.pop('contour')
        res[pop] = maps
        rxe, rxi = np.round(rxe), np.round(rxi)
        wxe, wxi = wx_info[pop]['xe'], wx_info[pop]['xi']
        wxe, wxi = np.round(wxe, 4), np.round(wxi, 4)
        rx_sel.append(
            [pop, r0, cv, rxe, rxi, wxe, wxi, xe_sec, xi_sec]
        )

        #break

# Save the results
with open(dirpath_out / 'rx_sel_info.json', 'w') as fid:
    json.dump(res, fid, indent=4, cls=json_utils.CustomEncoder)
rx_sel = pd.DataFrame(
    rx_sel,
    columns=[
        'pop', 'r0', 'cv', 'rxe', 'rxi',
        'wxe', 'wxi', 'xe_sec', 'xi_sec'
    ]
)
rx_sel.to_csv(dirpath_out / 'rx_sel.csv')

IT2
IT3
ITP4
ITS4
IT5A
IT5B
IT6
CT5A
CT5B
CT6
PT5B
PV2
PV3
PV4
PV5A
PV5B
PV6
SOM2
SOM3
SOM4
SOM5A
SOM5B
SOM6
VIP2
VIP3
VIP4
VIP5A
VIP5B
VIP6
NGF1
NGF2
NGF3
NGF4
NGF5A
NGF5B
NGF6


In [15]:
""" # Test read
proc_label = 'it2'
exp_name_sub_group = '2d_20x20_psp_0.5_xsec_soma_soma'
dirpath_out = dirpath_res / exp_name_sub_group / f'rxsel_{proc_label}'
df = pd.read_csv(dirpath_out / 'rx_sel.csv').set_index('pop')
df.drop(columns=['Unnamed: 0'], errors='ignore')
df['rxe'] = np.maximum(df['rxe'], 1e-3)
df['rxi'] = np.maximum(df['rxi'], 1e-3)
df.T.to_dict() """

" # Test read\nproc_label = 'it2'\nexp_name_sub_group = '2d_20x20_psp_0.5_xsec_soma_soma'\ndirpath_out = dirpath_res / exp_name_sub_group / f'rxsel_{proc_label}'\ndf = pd.read_csv(dirpath_out / 'rx_sel.csv').set_index('pop')\ndf.drop(columns=['Unnamed: 0'], errors='ignore')\ndf['rxe'] = np.maximum(df['rxe'], 1e-3)\ndf['rxi'] = np.maximum(df['rxi'], 1e-3)\ndf.T.to_dict() "